# Phase 2 — 블로그 & SNS 수집

## 수집 대상

| 소스 | 대상 | 방법 | 비고 |
|------|------|------|------|
| 네이버 블로그 | 한국인 라오스 여행객 EV 충전 경험 | requests + BeautifulSoup | 공개 블로그 |
| Facebook | LOCA EV, Green SM 공개 페이지 | Meta Graph API | ToS 준수 필수 |
| Instagram | 한국인 라오스 여행객 해시태그 | Meta Graph API | 공개 게시물만 |

⚠️ **Facebook/Instagram 주의**: Selenium 크롤링은 Meta ToS 위반 — Graph API 우선, 불가 시 수동 수집

In [ ]:
# !pip install requests beautifulsoup4 python-dotenv psycopg2-binary pandas

In [ ]:
import sys
sys.path.append('../')

import requests, time, re
import pandas as pd
from bs4 import BeautifulSoup
from dotenv import load_dotenv
import os

load_dotenv(dotenv_path='../.env')
from src.db import insert_df, fetch_df

---
## 1. 네이버 블로그 수집

네이버 검색 API 또는 BeautifulSoup으로 라오스 EV 관련 블로그 포스트를 수집합니다.

In [ ]:
# 네이버 검색 API 키 (https://developers.naver.com 에서 발급)
# .env에 추가: NAVER_CLIENT_ID, NAVER_CLIENT_SECRET
NAVER_CLIENT_ID     = os.getenv('NAVER_CLIENT_ID', '')
NAVER_CLIENT_SECRET = os.getenv('NAVER_CLIENT_SECRET', '')

# 수집 검색어
BLOG_QUERIES = [
    '라오스 전기차 충전',
    '라오스 EV 충전소',
    '라오스 여행 전기차',
    '비엔티안 전기차 충전',
    'LOCA EV 라오스',
    '라오스 EV 렌트',
]

In [ ]:
def search_naver_blog_api(query: str, display: int = 100) -> list:
    """
    네이버 검색 API로 블로그 포스트 검색
    API 키 없으면 빈 리스트 반환
    """
    if not NAVER_CLIENT_ID:
        print("  ⚠️ NAVER_CLIENT_ID 미설정 — BeautifulSoup 방식으로 전환")
        return []

    url = 'https://openapi.naver.com/v1/search/blog.json'
    headers = {
        'X-Naver-Client-Id':     NAVER_CLIENT_ID,
        'X-Naver-Client-Secret': NAVER_CLIENT_SECRET,
    }
    results = []
    start = 1
    while start <= min(1000, display):
        params = {'query': query, 'display': min(100, display - len(results)),
                  'start': start, 'sort': 'date'}
        try:
            resp = requests.get(url, headers=headers, params=params, timeout=10)
            items = resp.json().get('items', [])
            if not items: break
            results.extend(items)
            start += len(items)
            if len(items) < 100: break
            time.sleep(0.3)
        except Exception as e:
            print(f"  ❌ {e}")
            break
    return results


def scrape_blog_content(url: str) -> str:
    """블로그 URL에서 본문 텍스트 추출"""
    try:
        resp = requests.get(url, timeout=10,
                            headers={'User-Agent': 'Mozilla/5.0'})
        soup = BeautifulSoup(resp.text, 'html.parser')
        # 네이버 블로그 본문 영역
        content_el = (soup.find('div', {'class': 'se-main-container'}) or
                      soup.find('div', {'id': 'postViewArea'}) or
                      soup.find('div', {'class': 'post-view'}))
        if content_el:
            return content_el.get_text(separator=' ', strip=True)[:2000]
        return soup.get_text(separator=' ', strip=True)[:2000]
    except Exception:
        return ''

In [ ]:
# 네이버 블로그 수집 실행
all_blog_posts = []

for query in BLOG_QUERIES:
    print(f"\n🔍 '{query}' 검색 중...")
    items = search_naver_blog_api(query, display=100)

    for item in items:
        # HTML 태그 제거
        title   = re.sub('<[^>]+>', '', item.get('title', ''))
        snippet = re.sub('<[^>]+>', '', item.get('description', ''))
        link    = item.get('link', '')
        date_str = item.get('postdate', '')  # YYYYMMDD
        post_date = f"{date_str[:4]}-{date_str[4:6]}-{date_str[6:]}" if len(date_str) == 8 else None

        all_blog_posts.append({
            'platform':  'naver_blog',
            'title':     title,
            'content':   snippet,  # 요약본 (전문 수집 시 scrape_blog_content 활용)
            'author':    item.get('bloggername', ''),
            'post_date': post_date,
            'url':       link,
        })
    print(f"  → {len(items)}건 수집")
    time.sleep(0.5)

# 중복 제거 (URL 기준)
blog_df = pd.DataFrame(all_blog_posts).drop_duplicates('url').reset_index(drop=True)
blog_df = blog_df[blog_df['content'].str.strip().astype(bool)]

print(f"\n📊 네이버 블로그 총 수집: {len(blog_df)}건")
print(blog_df[['title', 'post_date']].head(10).to_string(index=False))

In [ ]:
# DB 저장
if not blog_df.empty:
    insert_df(blog_df, 'blog_posts')
    print(f"✅ blog_posts 테이블에 {len(blog_df)}건 저장")

---
## 2. Facebook 공개 페이지 수집 (Meta Graph API)

> ⚠️ **Meta ToS 준수**: 공개 페이지 게시물만 수집. Graph API 액세스 토큰 필요.

In [ ]:
# .env에 추가: FACEBOOK_ACCESS_TOKEN
FB_TOKEN = os.getenv('FACEBOOK_ACCESS_TOKEN', '')

# 수집 대상 페이지 (공개 페이지 ID 또는 username)
FB_PAGES = [
    {'page_id': 'LocaLaos',     'name': 'LOCA EV', 'country': 'LAO'},
    {'page_id': 'GreenSMLaos', 'name': 'Green SM Laos', 'country': 'LAO'},
]

In [ ]:
def scrape_fb_page(page_id: str, page_name: str, country: str,
                   token: str, limit: int = 100) -> pd.DataFrame:
    """
    Meta Graph API로 공개 페이지 게시물 수집
    토큰 없으면 빈 DataFrame 반환
    """
    if not token:
        print(f"  ⚠️ FACEBOOK_ACCESS_TOKEN 미설정 — {page_name} 건너뜀")
        return pd.DataFrame()

    url = f"https://graph.facebook.com/v19.0/{page_id}/posts"
    params = {
        'fields': 'message,created_time,story',
        'limit':  min(100, limit),
        'access_token': token,
    }
    rows = []
    try:
        while len(rows) < limit:
            resp = requests.get(url, params=params, timeout=10).json()
            if 'error' in resp:
                print(f"  ❌ API 오류: {resp['error']['message']}")
                break
            for post in resp.get('data', []):
                content = post.get('message') or post.get('story', '')
                if content:
                    rows.append({
                        'platform':  'facebook',
                        'page_name': page_name,
                        'country':   country,
                        'content':   content[:2000],
                        'post_date': post['created_time'][:10],
                        'collection_method': 'graph_api',
                    })
            paging = resp.get('paging', {})
            if 'next' not in paging: break
            url = paging['next']
            params = {}
            time.sleep(0.5)
    except Exception as e:
        print(f"  ❌ {e}")

    return pd.DataFrame(rows) if rows else pd.DataFrame()


# Facebook 수집 실행
all_fb = []
for page in FB_PAGES:
    print(f"\n📘 {page['name']} 페이지 수집 중...")
    df = scrape_fb_page(page['page_id'], page['name'], page['country'], FB_TOKEN)
    if not df.empty:
        all_fb.append(df)
        print(f"  → {len(df)}건")

fb_df = pd.concat(all_fb, ignore_index=True) if all_fb else pd.DataFrame()
print(f"\n📊 Facebook 총 수집: {len(fb_df)}건")

if not fb_df.empty:
    insert_df(fb_df, 'sns_posts')

---
## 3. 수집 결과 확인

In [ ]:
print("📊 blog_posts 현황")
print(fetch_df("""
    SELECT platform, COUNT(*) as 건수
    FROM blog_posts GROUP BY platform
""").to_string(index=False))

print("\n📊 sns_posts 현황")
print(fetch_df("""
    SELECT platform, page_name, country, COUNT(*) as 건수
    FROM sns_posts GROUP BY platform, page_name, country
""").to_string(index=False))